# Chapman ECG Classification with Optuna

Notebook này thực hiện:

1. Đọc dữ liệu Chapman từ file CSV.
2. Chia dữ liệu thành train, validation và test.
3. Huấn luyện mô hình `MultiScaleCNN`.
4. Tối ưu siêu tham số bằng Optuna.
5. Huấn luyện lại trên toàn bộ tập train.
6. Đánh giá Accuracy và Micro-F1 trên tập test.
7. Lưu siêu tham số tốt nhất vào `results_optuna.csv`.

> Đặt notebook sao cho đường dẫn `../data/chapman/csv/raw.csv` và các file `Chapman.py`, `MultiScaleCNN.py` có thể được import đúng.

## 1. Cài đặt thư viện

Bỏ dấu `#` nếu môi trường chưa cài các thư viện cần thiết.

In [ ]:
# %pip install torch pandas numpy scikit-learn optuna

## 2. Import thư viện

In [ ]:
import gc
import os
import random
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, random_split

from Chapman import ChapmanTestDataset, ChapmanTrainDataset
from Main import Main

C:\Users\nguye\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Thiết lập seed và thiết bị

In [2]:
def set_seed(seed_value: int = 42) -> None:
    random.seed(seed_value)
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    np.random.seed(seed_value)

    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4060


## 4. Đọc và chia dữ liệu

In [ ]:
DATA_PATH = Path("../data/chapman/csv/raw.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file dữ liệu: {DATA_PATH.resolve()}"
    )

data = pd.read_csv(DATA_PATH, header=None)

train_data_pd, test_data_pd = train_test_split(
    data,
    test_size=0.2,
    random_state=SEED,
)

data_train = ChapmanTrainDataset(train_data_pd)
data_test = ChapmanTestDataset(test_data_pd)

print("Total samples:", len(data))
print("Train + validation samples:", len(data_train))
print("Test samples:", len(data_test))

## 5. Chia tập train thành train và validation

In [ ]:
VAL_RATIO = 0.2

n_total = len(data_train)
n_val = int(n_total * VAL_RATIO)
n_train = n_total - n_val

split_generator = torch.Generator().manual_seed(SEED)

train_set, val_set = random_split(
    data_train,
    [n_train, n_val],
    generator=split_generator,
)

print("Training samples:", len(train_set))
print("Validation samples:", len(val_set))

## 6. Cấu hình huấn luyện

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

# Trên Windows/Jupyter, num_workers=0 thường ổn định hơn.
# Có thể tăng lên 2, 4 hoặc 8 nếu môi trường hỗ trợ.
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

## 7. Hàm tạo DataLoader

In [ ]:
def create_loader(dataset, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

## 8. Hàm huấn luyện một epoch

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY).long()

        optimizer.zero_grad(set_to_none=True)

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = running_loss / max(1, len(loader))
    accuracy = correct / max(1, total)

    return average_loss, accuracy

## 9. Hàm đánh giá

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_predictions = []
    all_labels = []

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY).long()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    average_loss = running_loss / max(1, len(loader))
    accuracy = correct / max(1, total)

    micro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="micro",
        zero_division=0,
    )

    return average_loss, accuracy, micro_f1

## 10. Hàm xây dựng mô hình

In [ ]:
def build_model(params: dict) -> Main:
    kernel_sizes_arr = list(map(int, params["kernel_sizes"].split("_")))
    return Main(
        kernel_sizes=kernel_sizes_arr,
        upcoming_kernel_size=params["upcoming_kernel_size"],
        out_channels=params["out_channels"],
        num_se_res_blocks=params["num_se_res_blocks"],
        reduction=params["reduction"],
        n_heads=params["n_heads"],
        attention_num_layers=params["attention_num_layers"],
        dropout_rate=params["dropout_rate"],
    ).to(device)

## 11. Optuna objective

In [ ]:
def objective(trial: optuna.Trial) -> float:
    model = None

    try:
        # Dùng cùng seed cho các trial để việc so sánh công bằng hơn.
        set_seed(SEED)

        params = {
            "lr": trial.suggest_float("lr", 1e-5, 1e-1, log=True),
            "batch_size": trial.suggest_categorical(
                "batch_size", [16, 32]
            ),
            "weight_decay": trial.suggest_float(
                "weight_decay", 1e-6, 1e-3, log=True
            ),
            "dropout_rate": trial.suggest_float(
                "dropout_rate", 0.1, 0.5
            ),
            "kernel_sizes": trial.suggest_categorical(
                "kernel_sizes", [  
                    "3_5_7",
                    "3_5_9",
                    "5_7_9",
                    "5_7_11",
                    "5_9_11",
                    "7_9_11",
                    "7_9_13",
                    "7_11_13",
                    "9_11_13",
                    "9_11_15",
                    "9_13_15"
                ]
            ),
            "upcoming_kernel_size": trial.suggest_categorical(
                "upcoming_kernel_size", [3, 5,7, 9]
            ),
            "out_channels": trial.suggest_categorical(
                "out_channels", [32, 64, 128, 256, 512, 1024]
            ),
            "num_se_res_blocks": trial.suggest_categorical(
                "num_se_res_blocks", [1, 2, 3, 4, 5,6,7,8]
            ),
            "reduction": trial.suggest_categorical(
                "reduction", [4, 8, 16]
            ),
            "n_heads": trial.suggest_categorical(
                "n_heads", [4, 8, 16, 32]
            ),
            "attention_num_layers": trial.suggest_categorical(
                "attention_num_layers", [1, 2, 3, 4, 5]
            ),
        }

        train_loader = create_loader(
            train_set,
            batch_size=params["batch_size"],
            shuffle=True,
        )

        val_loader = create_loader(
            val_set,
            batch_size=params["batch_size"],
            shuffle=False,
        )

        model = build_model(params)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=params["lr"],
            weight_decay=params["weight_decay"],
            betas=(0.9, 0.99),
        )

        max_epochs = 20
        best_val_micro_f1 = 0.0

        for epoch in range(max_epochs):
            train_loss, train_acc = train_one_epoch(
                model,
                train_loader,
                optimizer,
                loss_fn,
            )

            val_loss, val_acc, val_micro_f1 = evaluate(
                model,
                val_loader,
                loss_fn,
            )

            # Optuna tối ưu Micro-F1, đúng với nội dung in kết quả.
            trial.report(val_micro_f1, step=epoch)

            if trial.should_prune():
                raise optuna.TrialPruned()

            best_val_micro_f1 = max(
                best_val_micro_f1,
                val_micro_f1,
            )

            trial.set_user_attr(
                "last_epoch_metrics",
                {
                    "train_loss": float(train_loss),
                    "train_accuracy": float(train_acc),
                    "val_loss": float(val_loss),
                    "val_accuracy": float(val_acc),
                    "val_micro_f1": float(val_micro_f1),
                },
            )

        return best_val_micro_f1

    except torch.cuda.OutOfMemoryError:
        print(f"Trial {trial.number} bị CUDA Out Of Memory.")
        raise optuna.TrialPruned()

    finally:
        if model is not None:
            del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 12. Chạy Optuna study

In [ ]:
def run_study(n_trials: int = 25) -> optuna.Study:
    sampler = optuna.samplers.TPESampler(seed=SEED)

    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5,
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name="chapman_multiscale_cnn",
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=1,
        gc_after_trial=True,
    )

    print("Best validation Micro-F1:", study.best_value)
    print("Best parameters:")
    print(study.best_params)

    return study

Chạy cell dưới đây để bắt đầu tìm kiếm siêu tham số. Có thể giảm `N_TRIALS` khi muốn kiểm tra nhanh.

In [ ]:
N_TRIALS = 200

study = run_study(n_trials=N_TRIALS)
best_params = study.best_params

## 13. Xem kết quả các trial

In [ ]:
trials_df = study.trials_dataframe()
trials_df.head()

## 14. Huấn luyện mô hình cuối cùng và đánh giá test

In [ ]:
from sklearn.metrics import confusion_matrix

def safe_divide(a, b):
    return np.divide(
        a,
        b,
        out=np.zeros_like(a, dtype=float),
        where=b != 0
    )

def compute_metrics(all_labels, all_preds, class_names):
    num_classes = len(class_names)
    cm = confusion_matrix(all_labels, all_preds, labels=np.arange(num_classes))
    total_support = cm.sum()
    support = cm.sum(axis=1)

    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    TN = cm.sum() - (TP + FP + FN)

    precision = safe_divide(TP, TP + FP)
    recall = safe_divide(TP, TP + FN)
    specificity = safe_divide(TN, TN + FP)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    acc_per_class = safe_divide(TP + TN, TP + TN + FP + FN)

    weights = safe_divide(support, total_support)
    macro_precision = precision.mean()
    macro_recall = recall.mean()
    macro_specificity = specificity.mean()
    macro_f1 = f1.mean()

    weighted_precision = (precision * weights).sum()
    weighted_recall = (recall * weights).sum()
    weighted_specificity = (specificity * weights).sum()
    weighted_f1 = (f1 * weights).sum()
    
    overall_acc = TP.sum() / max(1, total_support)
    micro_precision = overall_acc
    micro_recall = overall_acc
    micro_f1 = overall_acc
    
    FP_s = FP.sum()
    TN_s = TN.sum()
    micro_specificity = TN_s / (TN_s + FP_s) if (TN_s + FP_s) > 0 else 0.0

    rows = []
    for i, name in enumerate(class_names):
        rows.append({
            "Class": name,
            "Accuracy": f"{acc_per_class[i] * 100:.2f}%",
            "Precision": f"{precision[i]:.4f}",
            "Recall": f"{recall[i]:.4f}",
            "Specificity": f"{specificity[i]:.4f}",
            "F1 Score": f"{f1[i]:.4f}",
            "Support": int(support[i])
        })
    
    for avg_name, p, r, s, f in [
        ("Macro avg", macro_precision, macro_recall, macro_specificity, macro_f1),
        ("Micro avg", micro_precision, micro_recall, micro_specificity, micro_f1),
        ("Weighted avg", weighted_precision, weighted_recall, weighted_specificity, weighted_f1)
    ]:
        rows.append({
            "Class": avg_name, "Accuracy": "", "Precision": f"{p:.4f}",
            "Recall": f"{r:.4f}", "Specificity": f"{s:.4f}", "F1 Score": f"{f:.4f}", "Support": int(total_support)
        })

    rows.append({
        "Class": "Overall", "Accuracy": f"{overall_acc * 100:.2f}%", "Precision": "",
        "Recall": "", "Specificity": "", "F1 Score": "", "Support": int(total_support)
    })

    return {
        "confusion_matrix": cm, "df_metrics": pd.DataFrame(rows), "overall_acc": overall_acc,
        "macro_precision": macro_precision, "macro_recall": macro_recall, "macro_specificity": macro_specificity, "macro_f1": macro_f1,
        "weighted_precision": weighted_precision, "weighted_recall": weighted_recall, "weighted_specificity": weighted_specificity, "weighted_f1": weighted_f1,
        "micro_precision": micro_precision, "micro_recall": micro_recall, "micro_specificity": micro_specificity, "micro_f1": micro_f1
    }

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Confusion matrix
def plot_confusion_matrix(cm, class_names, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[0], linewidths=0.5)
    axes[0].set_title("Confusion Matrix - Counts")
    axes[0].set_xlabel("Predicted Label")
    axes[0].set_ylabel("True Label")

    row_sum = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm.astype(float), row_sum, out=np.zeros_like(cm, dtype=float), where=row_sum != 0)

    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
    axes[1].set_title("Confusion Matrix - Normalized")
    axes[1].set_xlabel("Predicted Label")
    axes[1].set_ylabel("True Label")
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

# Metrics table
def plot_metrics_table(df_metrics, save_path):
    df_plot = df_metrics.copy()
    col_labels = list(df_plot.columns)
    cell_text = df_plot.values.tolist()
    n_rows = len(cell_text)

    fig, ax = plt.subplots(figsize=(14, 0.6 * n_rows + 1.8))
    ax.axis("off")
    table = ax.table(cellText=cell_text, colLabels=col_labels, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.15, 1.6)
    
    ax.set_title("Metrics Summary Table", fontsize=14, fontweight="bold", pad=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
def train_final_and_test(
    best_params: dict,
    num_epochs: int,
):
    print("Best parameters:")
    print(best_params)

    set_seed(SEED)

    batch_size = best_params["batch_size"]

    full_train_loader = create_loader(
        data_train,
        batch_size=batch_size,
        shuffle=True,
    )

    test_loader = create_loader(
        data_test,
        batch_size=batch_size,
        shuffle=False,
    )

    model = build_model(best_params)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        betas=(0.9, 0.99),
    )

    training_history = []

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model,
            full_train_loader,
            optimizer,
            loss_fn,
        )

        training_history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_accuracy": train_acc,
            }
        )

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc * 100:.2f}%"
        )

    test_loss, test_acc, test_micro_f1 = evaluate(
        model,
        test_loader,
        loss_fn,
    )

    
    CLASS_NAMES  = ["AFIB", "GSVT", "SB", "SR"]   # adjust if needed

    # Confusion matrix
    plot_confusion_matrix(
        compute_metrics(
            [label for _, label in data_test],
            [model(x.unsqueeze(0).to(device)).argmax(dim=1).item() for x, _ in data_test],
            class_names=CLASS_NAMES,
        )["confusion_matrix"],
        class_names=CLASS_NAMES,
        save_path="confusion_matrix.png",
    )
    
    plot_metrics_table(
        compute_metrics(
            [label for _, label in data_test],
            [model(x.unsqueeze(0).to(device)).argmax(dim=1).item() for x, _ in data_test],
            class_names=CLASS_NAMES,
        )["df_metrics"],
        save_path="metrics_summary_table.png",
    )
    
    cm_path = os.path.join("confusion_matrix.png")
    table_path = os.path.join("metrics_table.png")

    print(f"Confusion matrix: {cm_path}")
    print(f"Metrics summary table: {table_path}")

    result_row = {
        **best_params,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_micro_f1": test_micro_f1,
        "num_epochs": num_epochs,
    }

    pd.DataFrame([result_row]).to_csv(
        "results_optuna.csv",
        index=False,
    )

    pd.DataFrame(training_history).to_csv(
        "training_history.csv",
        index=False,
    )

    return model, pd.DataFrame(training_history), result_row

In [ ]:
final_model, history_df, final_result = train_final_and_test(
    best_params,
    num_epochs=80,
)

## 15. Xem kết quả cuối cùng

In [ ]:
pd.DataFrame([final_result])

## 16. Vẽ lịch sử huấn luyện

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"])
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss by Epoch")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history_df["epoch"],
    history_df["train_accuracy"] * 100,
)
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy (%)")
plt.title("Training Accuracy by Epoch")
plt.grid(True)
plt.show()